# Metabolomics overview plot w/ Sirius Canopus classifications

Date created: 12/29/2024

Notebook author: Yang Chen

Data analysis by: Britta De Pessemier

This notebook plots the following:
- Metabolomics overview plots showing number of total metabolites, with and without suspects library, and classified annotations w/ Sirius Canopus classifications

In [423]:
# Import Python packages
import pandas as pd
import numpy as np
from numpy import sqrt
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib_venn import venn3
import plotly.graph_objects as go
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib import colormaps
from upsetplot import from_indicators, plot
from upsetplot import from_memberships, UpSet

In [424]:
# Read in table of all spectra and output number of total metabolites
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')

total_num_metabolites = info_feature_complete.shape[0]
print(f'Total number of consensus MS2 spectra metabolites: ' + str(total_num_metabolites))

Total number of consensus MS2 spectra metabolites: 7683


In [425]:
# Read in GNPS2 metabolites table obtained WITHOUT suspects library
gnps_without_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps_withoutsuspects.tsv', sep='\t')

# Drop any without molecular formula
# gnps_without_suspects = gnps_without_suspects.dropna(subset=['molecular_formula'])

num_gnps_without_suspects = gnps_without_suspects.shape[0]
print(f'Number of metabolites matched with GNPS2 Standard Spectral Library: ' + str(num_gnps_without_suspects))

Number of metabolites matched with GNPS2 Standard Spectral Library: 432


In [426]:
# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')

# Drop any without molecular formula
# gnps_with_suspects = gnps_with_suspects.dropna(subset=['molecular_formula'])

num_gnps_with_suspects = gnps_with_suspects.shape[0]
print(f'Number of metabolites matched with GNPS2 Suspect Spectral Library: ' + str(num_gnps_with_suspects))

Number of metabolites matched with GNPS2 Suspect Spectral Library: 1027


In [427]:
# Read in Sirius Canopus classifications file
sirius_result = pd.read_csv('../Data/metabolomics/Run3_10252024/canopus_structure_summary.csv')
total_num_sirius = sirius_result.shape[0]
print(f'Number of total Sirius4 annotations: ' + str(total_num_sirius))

Number of total Sirius4 annotations: 5625


In [428]:
# Read in Sirius Canopus classifications file
sirius_result = pd.read_csv('../Data/metabolomics/Run3_10252024/canopus_structure_summary.csv')

# Drop any without molecular formula
sirius_result = sirius_result.dropna(subset=['molecularFormula'])

# Set threshold (reccomendation from Dorrestein lab)
prob = 0.75

# Filter Sirius result
sirius_include = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob) &
    (sirius_result['ClassyFire#class Probability'] >= prob) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob)
]


sirius_include

,formulaRank,molecularFormula,adduct,precursorFormula,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,...,ClassyFire#most specific class,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,featureId,overallFeatureQuality
2,1,C6H15NO3,[M + H]+,C6H16NO3+,Carbohydrates,0.504,Aminosugars and aminoglycosides,0.372,Aminosugars,0.334,...,"1,2-aminoalcohols",0.896,Organic compounds; Alcohols and polyols; Organ...,150.112,31,0.509,638448044789280881,637772593182489535,261,NaN
3,19,C8H16N6O8,[M - H2O + H]+,C8H15N6O7+,Carbohydrates,0.996,Aminosugars and aminoglycosides,0.537,Aminosugars,0.198,...,Pentoses,0.533,Organic compounds; Organoheterocyclic compound...,307.102,31,0.511,638448046974513380,637772589617330870,37,NaN
5,1,C8H16N4O3,[M + H]+,C8H17N4O3+,Amino acids and Peptides,0.980,Small peptides,0.997,Dipeptides,0.791,...,N-acyl-L-alpha-amino acids,0.566,Organic compounds; Lipids and lipid-like molec...,217.129,31,0.511,638448094017827270,637772603114602422,1013,NaN
8,8,C9H11N5O2,[M + H]+,C9H12N5O2+,Carbohydrates,0.662,Nucleosides,0.673,Purine alkaloids,0.720,...,6-aminopurines,0.653,Organic compounds; Organoheterocyclic compound...,222.097,31,0.513,638448142449456314,637772592301685625,216,NaN
9,1,C18H33NO15,[M + H]+,C18H34NO15+,Carbohydrates,1.000,Saccharides,0.994,Polysaccharides,0.660,...,Disaccharides,0.633,Organic compounds; Organoheterocyclic compound...,504.192,31,0.513,638448140373275749,637772603525644264,1061,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5617,1,C6H14N2,[M + H]+,C6H15N2+,Alkaloids,0.530,Ornithine alkaloids,0.258,Polyamines,0.584,...,Dialkylamines,0.670,Organic compounds; Organonitrogen compounds; D...,115.123,505,8.422,638475804086467823,637772835634264859,32752,NaN
5619,1,C6H9N3,[M + H]+,C6H10N3+,Alkaloids,0.962,Nicotinic acid alkaloids,0.176,Pyridine alkaloids,0.173,...,Aminopyrimidines and derivatives,0.855,Organic compounds; Organoheterocyclic compound...,124.087,505,8.425,638475779147132921,637772836515068834,32921,NaN
5620,2,C30H44O,[M + K]+,C30H44KO+,Shikimates and Phenylpropanoids,0.462,Flavonoids,0.266,Chalcones,0.230,...,Phenylpropanes,0.887,Organic compounds; Ketones; Organooxygen compo...,459.301,506,8.431,638475783681175681,637772836619926454,32963,NaN
5621,1,C30H38N2O2,[M + H3N + H]+,C30H42N3O2+,Alkaloids,0.252,Anthranilic acid alkaloids,0.096,Acridone alkaloids,0.162,...,Phenylpropanes,0.965,Organic compounds; Organic acids and derivativ...,476.328,506,8.434,638475786025792128,637772836590566321,32962,NaN


In [429]:
# Check number of metabolite features where Sirius ClassyFire#most specific class Probability >= 0.75 (reccomended from Dorrestein lab)
num_sirius_include = sirius_include.shape[0]
print(f'Number of metabolites from Sirius CANPOPUS ClassyFire#most specific class Probability >= 0.75: {num_sirius_include}')

Number of metabolites from Sirius CANPOPUS ClassyFire#most specific class Probability >= 0.75: 2136


In [430]:
# Plot a histogram of Sirius ClassyFire#most specific class Probability
plt.figure(figsize=(8, 6))
plt.hist(sirius_result['ClassyFire#most specific class Probability'], bins=50, edgecolor='black', alpha=0.7, color='#00305d')
plt.xlabel('Probability')
plt.ylabel('Frequency')
plt.title('Distribution of ClassyFire Most Specific Class Probability in Sirius CANOPUS', fontsize=12)
plt.axvline(x=0.75, color='red', linestyle='dashed', linewidth=1, label='Threshold = 0.75')
plt.text(0.86, plt.ylim()[1] * 0.88, f'Total annotations = {sirius_result.shape[0]}', fontsize=10, color='black')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig('../Figures/metabolomics_Figures/overview/Sirius_MPCprob_histogram.png', dpi=600)
plt.savefig('../Figures/metabolomics_Figures/overview/Sirius_MPCprob_histogram.svg')

In [431]:
# Concatenate gnps_without_suspects and gnps_with_suspects_gold
gnps_total = pd.concat([gnps_without_suspects, gnps_with_suspects], ignore_index=True)

gnps_total_unique_features = gnps_total.drop_duplicates(subset=['#Scan#'])
gnps_total_unique_features = gnps_total_unique_features.rename(columns={'#Scan#': 'FeatureId'})
gnps_final = gnps_total_unique_features[['FeatureId', 'superclass', 'class', 'subclass']]

# Filter GNPS result
gnps_final = gnps_final[
    gnps_final['superclass'].notna() &
    gnps_final['class'].notna() &
    gnps_final['subclass'].notna()
]


gnps_final

,FeatureId,superclass,class,subclass
0,67,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
1,2239,Organic oxygen compounds,Organooxygen compounds,Carbohydrates and carbohydrate conjugates
3,1368,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
4,1974,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
5,26200,Lipids and lipid-like molecules,Fatty Acyls,Fatty alcohols
...,...,...,...,...
531,5590,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
1002,25261,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
1056,32012,Lipids and lipid-like molecules,Fatty Acyls,Fatty amides
1116,1088,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"


In [432]:
sirius_include = sirius_include.rename(columns={'featureId': 'FeatureId'})
sirius_final = sirius_include[['FeatureId', 'ClassyFire#superclass', 'ClassyFire#class', 'ClassyFire#subclass']]

# Rename columns so they can be concatenated
sirius_final = sirius_final.rename(columns={'ClassyFire#superclass': 'superclass'})
sirius_final = sirius_final.rename(columns={'ClassyFire#class': 'class'})
sirius_final = sirius_final.rename(columns={'ClassyFire#subclass': 'subclass'})

# Drop rows in sirius_final if FeatureId is in gnps_final['FeatureID']
sirius_final = sirius_final[~sirius_final['FeatureId'].isin(gnps_final['FeatureId'])]

sirius_final

,FeatureId,superclass,class,subclass
2,261,Organic nitrogen compounds,Organonitrogen compounds,Amines
3,37,Organic oxygen compounds,Organooxygen compounds,Carbohydrates and carbohydrate conjugates
8,216,Organoheterocyclic compounds,Imidazopyrimidines,Purines and purine derivatives
9,1061,Organic oxygen compounds,Organooxygen compounds,Carbohydrates and carbohydrate conjugates
14,24,Organic oxygen compounds,Organooxygen compounds,Carbohydrates and carbohydrate conjugates
...,...,...,...,...
5610,33047,Organoheterocyclic compounds,Imidazopyrimidines,Purines and purine derivatives
5611,32914,Lipids and lipid-like molecules,Fatty Acyls,Fatty amides
5617,32752,Organic nitrogen compounds,Organonitrogen compounds,Amines
5619,32921,Organoheterocyclic compounds,Diazines,Pyrimidines and pyrimidine derivatives


In [433]:
# Find number of feature IDs added with each layer
print(f'Total number of consensus MS2 spectra metabolites: ' + str(total_num_metabolites))

print(f'Number of metabolites matched with GNPS2 Standard Spectral Library: ' + str(num_gnps_without_suspects))

gnps_total_features = gnps_total['#Scan#']
num_gnps_total_unique_features = gnps_total['#Scan#'].nunique()

print(f'Number of unique metabolites total from GNPS with Suspect Library: ' + str(num_gnps_total_unique_features))

# How many added from Sirius ClassyFire Most Specific Class prob >= 0.75
sirius_include_features = sirius_include['FeatureId']

# Concatenate gnps_final and sirius_final
metabolites_to_include = pd.concat([gnps_final, sirius_final], ignore_index=True)

print(f'Number of final unique features (including Sirius prob > 0.75): {metabolites_to_include.shape[0]}')


Total number of consensus MS2 spectra metabolites: 7683
Number of metabolites matched with GNPS2 Standard Spectral Library: 432
Number of unique metabolites total from GNPS with Suspect Library: 1032
Number of final unique features (including Sirius prob > 0.75): 2276


### Nested Venn Diagram showing number of metabolites recovered

In [453]:
# Define circle sizes
size1 = total_num_metabolites  # Size of Circle 1
size2 = total_num_sirius  # Number of total Sirius4 annotations
size3 = metabolites_to_include.shape[0]   # Size of Circle 2
size4 = num_gnps_total_unique_features   # Size of Circle 3
size5 = num_gnps_without_suspects   # Size of Circle 4

# Calculate the radii based on the square root of the sizes (scaled for visualization)
scaling_factor = 0.005  # Adjust this value for appropriate circle size in the plot

radius1 = np.sqrt(size1) * scaling_factor  # Radius for Circle 1
radius2 = np.sqrt(size2) * scaling_factor  # Radius for Circle 5
radius3 = np.sqrt(size3) * scaling_factor  # Radius for Circle 2
radius4 = np.sqrt(size4) * scaling_factor  # Radius for Circle 3
radius5 = np.sqrt(size5) * scaling_factor  # Radius for Circle 4

# Calculate the percentage for each circle relative to the total size
total_size = size1
percent1 = (size2 / size1) * 100
percent2 = (size3 / size1) * 100
percent3 = (size4 / size1) * 100
percent4 = (size5/ size1) * 100

# Create a figure
fig, ax = plt.subplots()

# Choose aesthetically pleasing colors (pastel-like colors)
circle1 = plt.Circle((0.5, 0.5), radius1, facecolor='#000000', alpha=0.6, edgecolor='black', linewidth=2, label=f'Total Unique MS2 Spectra (n={size1})')
circle2 = plt.Circle((0.5, 0.44), radius2, facecolor='#D3D3D3', alpha=0.6, edgecolor='black', linewidth=2, label=f'All Sirius4 CANOPUS Annotations (n={size2})')
circle3 = plt.Circle((0.5, 0.28), radius3, facecolor='#0096FF', alpha=0.6, edgecolor='black', linewidth=2, label=f'Annotated GNPS Matches and Sirius4 CANOPUS\nfiltered by MostSpecificClass probability >= 75% (n={size3})')
circle4 = plt.Circle((0.5, 0.22), radius4, facecolor='#ADD8E6', alpha=0.6, edgecolor='black', linewidth=2, label=f'GNPS Standard and Suspect Spectral Libraries (n={num_gnps_total_unique_features})')
circle5 = plt.Circle((0.5, 0.17), radius5, facecolor='#00008B', alpha=0.6, edgecolor='black', linewidth=2, label=f'GNPS Standard Spectral Library (n={num_gnps_without_suspects})')

# Add the circles to the plot
ax.add_patch(circle1)
ax.add_patch(circle2)
ax.add_patch(circle3)
ax.add_patch(circle4)
ax.add_patch(circle5)

# Add percentage text inside the circles
ax.text(0.505, 0.66, f'{percent1:.1f}%', ha='center', va='center', fontsize=10, color='white')
ax.text(0.505, 0.44, f'{percent2:.1f}%', ha='center', va='center', fontsize=10, color='white')
ax.text(0.505, 0.3, f'{percent3:.1f}%', ha='center', va='center', fontsize=10, color='black')
ax.text(0.505, 0.135, f'{percent4:.1f}%', ha='center', va='center', fontsize=10, color='white')

# Set limits to ensure all circles are fully visible
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Set aspect ratio to be equal to ensure circles are circular
ax.set_aspect('equal')

# Remove all borders
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove ticks and labels
ax.set_xticks([])
ax.set_yticks([])

# Add legend for the circles with their respective sizes and place it outside the plot
ax.legend(loc='upper left', bbox_to_anchor=(-0.3, 0), fontsize=10)

# Add title with better font style and size
plt.title("Metabolites Identified from Workflow", loc='center', fontsize=16)

# Adjust layout to make room for the legend
plt.tight_layout()

# Save figure
plt.savefig('../Figures/metabolomics_Figures/overview/ms2_venn_diagram_final.png', dpi=600)

In [435]:
# # Define circle sizes
# size1 = total_num_metabolites  # Total Unique MS2 Spectra
# size2 = metabolites_to_include.shape[0]  # Annotated GNPS Matches and Sirius4 CANOPUS
# size3 = num_gnps_total_unique_features  # GNPS Standard and Suspect Spectral Libraries
# size4 = num_gnps_without_suspects  # GNPS Standard Spectral Library
# size5 = total_num_sirius  # Number of total Sirius4 annotations

# # Calculate the radii based on the square root of the sizes (scaled for visualization)
# scaling_factor = 0.005  # Adjust this value for appropriate circle size in the plot

# radius1 = np.sqrt(size1) * scaling_factor  # Radius for Circle 1
# radius2 = np.sqrt(size2) * scaling_factor  # Radius for Circle 2
# radius3 = np.sqrt(size3) * scaling_factor  # Radius for Circle 3
# radius4 = np.sqrt(size4) * scaling_factor  # Radius for Circle 4
# radius5 = np.sqrt(size5) * scaling_factor  # Radius for Circle 5

# # Calculate the percentage for each circle relative to the total size
# total_size = size1
# percent1 = (size2 / size1) * 100
# percent2 = (size3 / size1) * 100
# percent3 = (size4 / size1) * 100
# percent4 = (size5 / size1) * 100

# # Create a figure
# fig, ax = plt.subplots(figsize=(6, 6))

# # Define circle positions and colors
# circles = [
#     (0.5, 0.5, radius1, '#000000', 'Total Unique MS2 Spectra', size1),
#     (0.5, 0.3, radius2, '#0096FF', 'Annotated GNPS Matches and Sirius4 CANOPUS', size2),
#     (0.5, 0.22, radius3, '#ADD8E6', 'GNPS Standard and Suspect Spectral Libraries', size3),
#     (0.5, 0.16, radius4, '#00008B', 'GNPS Standard Spectral Library', size4),
#     (0.5, 0.08, radius5, '#FFA07A', 'Number of total Sirius4 annotations', size5)
# ]

# # Add the circles to the plot
# for x, y, r, color, label, size in circles:
#     circle = plt.Circle((x, y), r, facecolor=color, alpha=0.6, edgecolor='black', linewidth=2, label=f'{label} (n={size})')
#     ax.add_patch(circle)

# # Add percentage text inside the circles
# ax.text(0.505, 0.44, f'{percent1:.1f}%', ha='center', va='center', fontsize=10, color='white')
# ax.text(0.505, 0.3, f'{percent2:.1f}%', ha='center', va='center', fontsize=10, color='black')
# ax.text(0.505, 0.135, f'{percent3:.1f}%', ha='center', va='center', fontsize=10, color='white')
# ax.text(0.505, 0.06, f'{percent4:.1f}%', ha='center', va='center', fontsize=10, color='black')

# # Set limits to ensure all circles are fully visible
# ax.set_xlim(0, 1)
# ax.set_ylim(0, 1)

# # Set aspect ratio to be equal to ensure circles are circular
# ax.set_aspect('equal')

# # Remove all borders
# for spine in ax.spines.values():
#     spine.set_visible(False)

# # Remove ticks and labels
# ax.set_xticks([])
# ax.set_yticks([])

# # Add legend for the circles with their respective sizes and place it outside the plot
# ax.legend(loc='upper left', bbox_to_anchor=(-0.25, 0), fontsize=12)

# # Add title with better font style and size
# plt.title("Metabolites Identified from Workflow", loc='center', fontsize=16)

# # Adjust layout to make room for the legend
# plt.tight_layout()

# # Save figure
# plt.savefig('../Figures/metabolomics_Figures/overview/ms2_venn_diagram_final_sirius-all.png', dpi=600)

# # Display the plot
# plt.show()


### Sankey plot with final metabolites

In [436]:
# Calculate value counts for each column
superclass_order = metabolites_to_include['superclass'].value_counts().index.tolist()
class_order = metabolites_to_include.groupby('superclass')['class'].value_counts().index.get_level_values(1).tolist()
subclass_order = metabolites_to_include.groupby(['superclass', 'class'])['subclass'].value_counts().index.get_level_values(2).tolist()

# Convert columns into ordered categorical data
metabolites_to_include['superclass'] = pd.Categorical(metabolites_to_include['superclass'], categories=superclass_order, ordered=True)
metabolites_to_include['class'] = pd.Categorical(metabolites_to_include['class'], categories=class_order, ordered=True)
metabolites_to_include['subclass'] = pd.Categorical(metabolites_to_include['subclass'], categories=subclass_order, ordered=True)

# Sort the DataFrame by the hierarchical order
metabolites_to_include = metabolites_to_include.sort_values(by=['superclass', 'class', 'subclass'])
metabolites_to_include

,FeatureId,superclass,class,subclass
0,67,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
2,1368,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
3,1974,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
5,1119,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
22,1017,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues"
...,...,...,...,...
99,19463,Organophosphorus compounds,Organic phosphines and derivatives,Organophosphine oxides
100,1466,Organophosphorus compounds,Organic phosphines and derivatives,Organophosphine oxides
584,2828,"Organic 1,3-dipolar compounds","Allyl-type 1,3-dipolar organic compounds",Organic nitro compounds
1249,13302,"Organic 1,3-dipolar compounds","Allyl-type 1,3-dipolar organic compounds",Organic nitro compounds


In [437]:
superclass_order

['Organic acids and derivatives',
 'Organic oxygen compounds',
 'Lipids and lipid-like molecules',
 'Benzenoids',
 'Organic nitrogen compounds',
 'Organoheterocyclic compounds',
 'Phenylpropanoids and polyketides',
 'Lignans, neolignans and related compounds',
 'Organosulfur compounds',
 'Nucleosides, nucleotides, and analogues',
 'Organophosphorus compounds',
 'Organic 1,3-dipolar compounds',
 'Organometallic compounds']

In [438]:
starting_metabolites = metabolites_to_include.shape[0]

# Count occurrences for each unique label in superclass, class, and subclass
superclass_counts = metabolites_to_include['superclass'].value_counts()
class_counts = metabolites_to_include['class'].value_counts()
subclass_counts = metabolites_to_include['subclass'].value_counts()

# Set threshold as 1% of total included here
threshold = metabolites_to_include.shape[0] * 0.01

# Filter out categories with fewer than 10 occurrences in class and subclass
metabolites_to_include = metabolites_to_include[
    metabolites_to_include['superclass'].isin(superclass_counts[superclass_counts >= threshold].index) &
    metabolites_to_include['class'].isin(class_counts[class_counts >= threshold].index) &
    metabolites_to_include['subclass'].isin(subclass_counts[subclass_counts >= threshold].index)
]

# Recalculate unique labels after collapsing class and subclass
superclass_labels = metabolites_to_include['superclass'].unique()  # All categories stay in superclass
class_labels = metabolites_to_include['class'].unique()  # Collapsed classes
subclass_labels = metabolites_to_include['subclass'].unique()  # Collapsed subclasses

# Sort superclass categories by their counts in descending order
sorted_superclass_counts = superclass_counts.sort_values(ascending=False)

# Get the sorted order of superclass labels
sorted_superclass_labels = sorted_superclass_counts.index.tolist()

# Recalculate y positions based on sorted superclass labels
superclass_y = [i / len(sorted_superclass_labels) for i in range(len(sorted_superclass_labels))]

# Assign x positions for superclass, class, and subclass categories
superclass_x = [0.5] * len(sorted_superclass_labels)
class_y = [i / len(class_labels) for i in range(len(class_labels))]
class_x = [0.6] * len(class_labels)
subclass_y = [i / len(subclass_labels) for i in range(len(subclass_labels))]
subclass_x = [0.9] * len(subclass_labels)  # Subclass nodes at 90% width

# Combine all positions
node_x = [0] + superclass_x + class_x + subclass_x + [1]  # Include root and unannotated node positions
node_y = [0.5] + superclass_y + class_y + subclass_y + [0.5]  # Include root and unannotated node positions

# Update the label order to match the sorted superclass labels
# sorted_all_labels = [f'Annotated Metabolites\nn={starting_metabolites}'] + sorted_superclass_labels + list(class_labels) + list(subclass_labels)
sorted_all_labels = [f'Annotated Metabolites<br>n={starting_metabolites}'] + sorted_superclass_labels + list(class_labels) + list(subclass_labels)


# Create a dictionary to map labels to indices
label_dict = {label: idx for idx, label in enumerate(sorted_all_labels)}

# Generate source, target, and value lists for the Sankey plot
sources = []
targets = []
values = []

# 'GNPS2 identified metabolites (without suspect library)' -> Superclass
for idx, row in metabolites_to_include.iterrows():
    source_idx = label_dict[f'Annotated Metabolites<br>n={starting_metabolites}']
    superclass_idx = label_dict[row['superclass']]
    sources.append(source_idx)
    targets.append(superclass_idx)
    values.append(1)  # Assuming one metabolite per row, otherwise adjust the value

# Superclass -> Class
for idx, row in metabolites_to_include.iterrows():
    superclass_idx = label_dict[row['superclass']]
    class_idx = label_dict[row['class']]
    sources.append(superclass_idx)
    targets.append(class_idx)
    values.append(1)  # Assuming one metabolite per row

# Class -> Subclass
for idx, row in metabolites_to_include.iterrows():
    class_idx = label_dict[row['class']]
    subclass_idx = label_dict[row['subclass']]
    sources.append(class_idx)
    targets.append(subclass_idx)
    values.append(1)  # Again, assuming one metabolite per row

# Generate y positions for nodes based on their category
superclass_y = [i / len(superclass_labels) for i in range(len(superclass_labels))]
class_y = [i / len(class_labels) for i in range(len(class_labels))]
subclass_y = [i / len(subclass_labels) for i in range(len(subclass_labels))]

# Generate x positions for each category of nodes
superclass_x = [0.3] * len(superclass_labels)  # Superclass nodes at 30% width
class_x = [0.6] * len(class_labels)            # Class nodes at 60% width
subclass_x = [1] * len(subclass_labels)      # Subclass nodes at 90% width

# Combine all positions
node_x = [0] + superclass_x + class_x + subclass_x + [2]  # Add positions for the root and unannotated node
node_y = [0.5] + superclass_y + class_y + subclass_y + [0.5]  # Add positions for the root and unannotated node

# Get the 'tab10' colormap
tab10 = colormaps['tab10']

# Generate colors starting from the last color and cycling forward
tab10_shifted = [tab10((i + 9) % 10) for i in range(10)]  # Do NOT overwrite tab10

# Assign colors to superclasses
superclass_palette = {label: tab10_shifted[i % 10] for i, label in enumerate(sorted_superclass_labels)}
superclass_colors = {label: mcolors.to_hex(superclass_palette[label]) for label in sorted_superclass_labels}

# Generate node colors
node_colors = []
for label in sorted_all_labels:
    if label in superclass_labels:
        # Use original superclass color
        node_colors.append(superclass_colors[label])
    elif label in class_labels:
        # Derive a lighter shade from its corresponding superclass
        superclass = metabolites_to_include[metabolites_to_include['class'] == label].iloc[0]['superclass']
        node_colors.append(mcolors.to_hex(mcolors.to_rgba(superclass_colors[superclass], alpha=0.8)))
    elif label in subclass_labels:
        # Derive an even lighter shade from the corresponding class
        related_class = metabolites_to_include[metabolites_to_include['subclass'] == label].iloc[0]['class']
        related_superclass = metabolites_to_include[metabolites_to_include['subclass'] == label].iloc[0]['superclass']
        node_colors.append(mcolors.to_hex(mcolors.to_rgba(superclass_colors[related_superclass], alpha=0.6)))
    else:
        # Default color for unannotated nodes
        node_colors.append('#ededed')


# Create the Sankey plot with spaced-out nodes and consistent colors
fig = go.Figure(go.Sankey(
    node=dict(
        pad=100,
        thickness=20,
        line=dict(color='black', width=0.5),
        label=sorted_all_labels,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values *2,
        color='rgba(58, 158, 224, 0.3)'  #3a9ee0 blue color
    )
))

fig.update_layout(
    title=dict(
        text="Hierarchical Classifications of Annotated Metabolites",
        x=0.5,  # Center the title
        xanchor='center',  # Align the title by its center
        font=dict(
            size=20,  # Adjust title font size
            color='black'  # Set the title text color (your desired color)
        )
    ),
    font=dict(size=13),  # General font size for other elements
    width=1000,
    height=600,
    margin=dict(l=10, r=225, t=50, b=10)
)

# Save the figure as PNG
fig.write_image("../Figures/metabolomics_Figures/overview/metabolomics_sankey_plot_final.png", format="png", width=1000, height=500, scale=2)

# Save the figure as SVG
fig.write_image("../Figures/metabolomics_Figures/overview/metabolomics_sankey_plot_final.svg", format="svg", width=1000, height=500)

fig.show()